# Live Tracker — 医药生物板块周度预测
## 每周运行一次，更新预测信号

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd; import numpy as np
import joblib; from pathlib import Path
from datetime import datetime
import warnings; warnings.filterwarnings('ignore')

TODAY = datetime.now().strftime('%Y-%m-%d')
print(f'Update: {TODAY}')

In [ ]:
# 1. 加载模型
MODEL_DIR = Path('../data/processed')
clf = joblib.load(MODEL_DIR / 'medical_classifier.pkl')
reg = joblib.load(MODEL_DIR / 'medical_regressor.pkl')
scaler = joblib.load(MODEL_DIR / 'medical_scaler.pkl')
screened_factors = joblib.load(MODEL_DIR / 'screened_factors.pkl')
feature_names = joblib.load(MODEL_DIR / 'feature_names.pkl')
meta = joblib.load(MODEL_DIR / 'model_meta.pkl')

print(f'Model: {meta["n_factors"]} factors, trained on {meta["train_weeks"]} weeks')
print(f'Test accuracy: {meta["test_accuracy"]:.1%}')
print(f'Screened factors: {screened_factors}')

In [ ]:
# 2. 拉取最新数据
from src.data_fetcher.akshare_source import AKShareSource
from src.data_fetcher.fred_source import FREDSource
from src.factors.factor_pipeline import FactorPipeline, align_factors_with_target
from src.features.engineer import FeatureEngineer

data = {}
data.update(AKShareSource().fetch_all('2018-01-01'))
try:
    data.update(FREDSource().fetch_all('2018-01-01'))
except Exception as e:
    print(f'FRED update failed (non-critical): {e}')

# 计算因子
factors_df = FactorPipeline().compute_all(data, target='medical', verbose=False)
target_price = data['sw_medical'].set_index('date')['close'].sort_index()
X, y = align_factors_with_target(factors_df, target_price, shift_target=1)

# 特征工程
X_sub = X[screened_factors]
fe = FeatureEngineer(lags=[1, 2, 4], rolling_windows=[4, 13], fill_method='ffill')
X_fe = fe.fit_transform(X_sub)

# 对齐列
for c in set(feature_names) - set(X_fe.columns):
    X_fe[c] = 0
X_fe = X_fe[feature_names]

# 标准化
X_scaled = scaler.transform(X_fe)

print(f'{len(X_scaled)} weeks of factor data ready')

In [ ]:
# 3. 预测
latest = X_scaled[-1:]
latest_date = X_fe.index[-1]

direction = clf.predict(latest)[0]
proba = clf.predict_proba(latest)[0]
magnitude = reg.predict(latest)[0]

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
color = 'green' if direction == 1 else 'red'
emoji = 'UP' if direction == 1 else 'DOWN'
ax.barh([0], [proba[1]], color=color, alpha=0.6)
ax.barh([0], [1], color='lightgray', alpha=0.3)
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xlabel('Probability')
ax.set_title(f'Prediction: {emoji} (prob={proba[1]:.0%}) | Expected: {magnitude:+.2f}% | {latest_date.date()} -> {(latest_date + pd.Timedelta(weeks=1)).date()}')
plt.tight_layout()
plt.show()

In [ ]:
# 4. 因子快照
latest_factors = factors_df[screened_factors].iloc[-1]
prev_factors = factors_df[screened_factors].iloc[-2]
snapshot = pd.DataFrame({'Latest': latest_factors, 'Change': latest_factors - prev_factors})
display(snapshot)

In [ ]:
# 5. 保存信号
history_path = Path('../data/processed/signal_history.csv')
new_row = pd.DataFrame([{
    'date': latest_date.date(),
    'direction': emoji,
    'prob_up': f'{proba[1]:.4f}',
    'expected_return': f'{magnitude:+.2f}%',
}])

if history_path.exists():
    hist = pd.read_csv(history_path)
    if str(latest_date.date()) not in hist['date'].values:
        hist = pd.concat([hist, new_row], ignore_index=True)
else:
    hist = new_row

hist.to_csv(history_path, index=False)
print(f'Signal history: {len(hist)} entries')
display(hist.tail(10))